# Round 6: fix the weak-student thinking-mode problem, evaluate the teachers, add main-pair seeds

Needs GPU T4 x2, Internet, and the `HF_TOKEN` secret ticked. Set `PART` in the first cell; the two
parts can run on two accounts at the same time.

- **`PART = "weak"`** (about 10 h): Qwen3-1.7B is a hybrid "thinking" model; before 2026-10-03 it
  was trained and evaluated with thinking on, so it could open every reply with a long `<think>`
  block and run out of the 256-token budget before any tool call. The code now turns thinking off
  in training and eval (a no-op for Qwen3-4B-Instruct-2507 and every OLMo-2 model). This part
  re-trains and re-evaluates the second pair (base, SFT, Self-distilled, Distilled-8B, seeds 0-2)
  into `pair2t_*`, and evaluates two teachers (Qwen3-14B, OLMo-2-13B) on the primary set into
  `teacher_eval_*`. `RUN_WEAK_DIAL = True` also re-runs the 0.05 dial point (+3.5 h).
- **`PART = "seeds"`** (about 9.5 h): main pair, seeds 5-9, SFT and Self-distilled, each evaluated
  on the primary AND extended sets (`sweep_main_baseline_<cond>.json`), to give the anchoring
  contrast more statistical power.

Finished jobs are skipped (checked on Hugging Face), so a stopped run can simply be repeated.

In [ ]:
PART = "weak"            # "weak" or "seeds" (use a different account for each)
RUN_WEAK_DIAL = False    # weak part only: also re-run the second pair's 0.05 dial point (+3.5 h)
EXTRA_SEEDS = [5, 6, 7, 8, 9]
MAX_MINUTES = 330        # per phase: stop starting new jobs after this long

In [ ]:
import base64
import os
import subprocess
import sys

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Kaggle secrets (Add-ons -> Secrets), all optional:
#   GH_TOKEN  read access to the GitHub repo, needed only while the repo is private
#   HF_TOKEN  write access to a Hugging Face repo, needed only to resume across sessions
os.environ.setdefault('ADBENCH_HF_REPO', 'NahlaNabil/adbench-run')
os.environ.setdefault('ADBENCH_RUN_TAG', 'v1-fixed')
try:
    from kaggle_secrets import UserSecretsClient
    _secrets = UserSecretsClient()
    for _name in ('GH_TOKEN', 'HF_TOKEN'):
        try:
            os.environ[_name] = _secrets.get_secret(_name)
        except Exception:
            pass
except Exception:
    pass


def git(*args, timeout=600):
    """Run git without ever prompting (a credentials prompt would hang an unattended run for
    hours). Uses GH_TOKEN when set; if that fails (revoked token, or a public repo that needs
    none) it retries once without it."""
    env = {**os.environ, 'GIT_TERMINAL_PROMPT': '0'}
    token = os.environ.get('GH_TOKEN')
    for use_token in ([True, False] if token else [False]):
        cmd = ['git']
        if use_token:
            basic = base64.b64encode(f"x-access-token:{token}".encode()).decode()
            cmd += ['-c', f'http.https://github.com/.extraheader=AUTHORIZATION: basic {basic}']
        try:
            subprocess.run(cmd + list(args), check=True, timeout=timeout, env=env)
            return
        except subprocess.CalledProcessError:
            if not use_token:
                raise
            print('git with GH_TOKEN failed; retrying without it.')


def run_module(*args, timeout=4 * 3600):
    """Run `python -m <args>` in a fresh process, print the tail of its output, and raise if it
    fails or exceeds `timeout` seconds (a bare `!` command never stops the notebook)."""
    proc = subprocess.run(
        [sys.executable, '-m', *args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, timeout=timeout
    )
    print(proc.stdout[-20000:])
    proc.check_returncode()


ON_KAGGLE = os.path.isdir('/kaggle')
REPO_DIR = '/kaggle/working/agentic-distillation-benchmark' if ON_KAGGLE else '/content/agentic-distillation-benchmark'

if not os.path.isdir(REPO_DIR):
    git('clone', 'https://github.com/Nahla-Nabil/agentic-distillation-benchmark.git', REPO_DIR)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements-colab.txt'], check=True, timeout=1800)

src_path = os.path.join(REPO_DIR, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)
os.environ['PYTHONPATH'] = src_path + os.pathsep + os.environ.get('PYTHONPATH', '')

In [ ]:
# re-sync to the latest commit
git('-C', REPO_DIR, 'checkout', '--', '.')
git('-C', REPO_DIR, 'pull')

## 1. Data (the default split, needed by every training job)

In [ ]:
run_module("adbench.data.prepare", "--config", "configs/data.yaml")

## 2. Check the token can write

In [ ]:
import os
if not os.environ.get("HF_TOKEN"):
    raise SystemExit("HF_TOKEN is not set: tick it under Add-ons -> Secrets for this notebook, then run again.")
os.environ["ADBENCH_RUN_TAG"] = "v2-seed0"
from adbench import pipeline_state as ps
ps.check_upload()

## 3. Run

Teacher-free jobs run as two GPU-pinned workers in parallel; jobs that load a teacher run
sequentially in one process with both GPUs visible (same reason as notebooks 14-19).

In [ ]:
import subprocess
import sys
import threading

assert PART in ("weak", "seeds"), PART


def pump(proc, name):
    for line in proc.stdout:
        if "Loading weights" in line or "it/s]" in line:
            continue
        print(f"[{name}] {line.rstrip()}", flush=True)


def start(module, jobs, extra, tag, gpu=None):
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    if gpu is not None:
        env["CUDA_VISIBLE_DEVICES"] = str(gpu)
    proc = subprocess.Popen(
        [sys.executable, "-m", module, "--jobs", ",".join(jobs), "--work-dir", f"/kaggle/working/r6_{tag}",
         "--max-minutes", str(MAX_MINUTES), *extra],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
    )
    thread = threading.Thread(target=pump, args=(proc, tag), daemon=True)
    thread.start()
    return proc, thread


def wait(workers, label):
    for proc, thread in workers:
        proc.wait()
        thread.join(timeout=30)
    codes = [proc.returncode for proc, _ in workers]
    print(label, "exit codes:", codes)
    if any(codes):
        raise RuntimeError(f"a {label} worker failed; see the log above")


def parallel(module, jobs, extra, label):
    if not jobs:
        return
    print(f"### {label}: {len(jobs)} jobs on two pinned GPUs ###")
    wait([start(module, jobs[g::2], extra, f"{label}_gpu{g}", gpu=g) for g in (0, 1) if jobs[g::2]], label)


def sequential(module, jobs, extra, label):
    if not jobs:
        return
    print(f"### {label}: {len(jobs)} jobs, sequential, both GPUs ###")
    wait([start(module, jobs, extra, label)], label)


if PART == "weak":
    parallel("adbench.evaluation.teacher_eval_worker", ["0:teacher", "0:teacher_olmo13"], [], "teachers")
    parallel("adbench.evaluation.second_pair_worker",
             ["0:base"] + [f"{s}:sft_only" for s in (0, 1, 2)], ["--no-think"], "weakA")
    sequential("adbench.evaluation.second_pair_worker",
               [f"{s}:{c}" for c in ("self_distill_small", "distilled_8b") for s in (0, 1, 2)], ["--no-think"], "weakB")
    if RUN_WEAK_DIAL:
        sequential("adbench.evaluation.sweep_worker",
                   [f"{s}:smallnt:sft_vheavy:self_distill_small" for s in (0, 1, 2)], [], "weakDial")
else:
    parallel("adbench.evaluation.sweep_worker",
             [f"{s}:main:baseline:sft_only" for s in EXTRA_SEEDS], ["--with-ext"], "seedsA")
    sequential("adbench.evaluation.sweep_worker",
               [f"{s}:main:baseline:self_distill" for s in EXTRA_SEEDS], ["--with-ext"], "seedsB")

## Summary

In [ ]:
import glob
import json

import pandas as pd

rows = []
for path in glob.glob("/kaggle/working/r6_*/results/*.json"):
    rows += json.load(open(path, encoding="utf-8"))
if rows:
    df = pd.DataFrame(rows)
    df["c35"] = df.chain_length.isin([3, 5])
    print(df.groupby(["task_set", "condition", "seed"]).success.mean().round(3))
    print(df[df.c35].groupby(["task_set", "condition"]).success.mean().round(3))
else:
    print("no result files in this session (everything was already on Hugging Face)")